# Harvard FairVLMed-10k Preprocessing Pipeline

This notebook prepares the **Harvard FairVLMed-10k** dataset for the benchmark experiments in:
> *The Impact of Adaptation on Fairness in Medical Multimodal Foundation Models*

### Overview
- **Modality**: Scanning Laser Ophthalmoscopy (SLO) Fundus Imaging (10,000 images)
- **Target Task**: Glaucoma detection (binary: `normal` vs. `glaucoma`)
- **Splits**: 7,000 train / 1,000 validation / 2,000 test
- **Demographic Attributes Evaluated**:
  1. **Sex**: Female vs. Male
  2. **Age**: Binned into quartiles (Q1: <51.9, Q2: 51.9-64.1, Q3: 64.1-73.0, Q4: >=73.0) & binary (>60)
  3. **Race**: White vs. Non-White (Asian, Black, etc.)
  4. **Language**: English vs. Non-English
  5. **Ethnicity**: Hispanic vs. Non-Hispanic
  6. **Marital Status**: Married/Partnered vs. Unmarried


In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd

# Configure paths
DATA_DIR = "./data/FairVLMed10k"
METADATA_PATH = os.path.join(DATA_DIR, "data_summary.csv")
ZIP_PATH = os.path.join(DATA_DIR, "dataset.zip")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Target data directory: {DATA_DIR}")

## 1. Extract Images (if zip provided)

In [ ]:
if os.path.exists(ZIP_PATH):
    print(f"Extracting {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Extraction complete!")
else:
    print("Zip file not found or already extracted. Continuing with CSV preprocessing.")

## 2. Load Metadata and Structure Labels

In [ ]:
if os.path.exists(METADATA_PATH):
    df = pd.read_csv(METADATA_PATH)
else:
    # Fallback to train/val/test if data_summary.csv is not present
    print(f"{METADATA_PATH} not found. Please place official data_summary.csv in {DATA_DIR}")
    df = pd.DataFrame()

print(f"Loaded {len(df)} records.")
if not df.empty:
    print(df.head())

## 3. Process Target Condition & Demographic Attributes

In [ ]:
if not df.empty:
    # Binary target label (Glaucoma: 1, Normal: 0)
    df['label'] = (df['glaucoma'].str.lower() == 'yes').astype(int)
    
    # Path to fundus image (relative to image_train_path)
    df['path'] = df.apply(
        lambda r: f"{str(r['use']).capitalize()}/slo_fundus_{str(r['filename'])[5:10]}.jpg",
        axis=1
    )
    
    # Clean string attributes
    df['gender'] = df['gender'].astype(str).str.lower().str.strip()
    df['race'] = df['race'].astype(str).str.lower().str.strip()
    df['language'] = df['language'].astype(str).str.lower().str.strip()
    df['ethnicity'] = df['ethnicity'].astype(str).str.lower().str.strip()
    df['maritalstatus'] = df['maritalstatus'].astype(str).str.lower().str.strip()
    
    # Age quartiles matching paper's exact splits: [9.6, 51.9, 64.1, 73.0, 99.8]
    df['age_group'] = pd.cut(
        df['age'],
        bins=[-np.inf, 51.92, 64.08, 73.02, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)
    
    # Senior vs non-senior
    df['is_senior'] = (df['age'] > 60).astype(int)
    
    print("Demographic distributions:")
    print("\nGender:", df['gender'].value_counts())
    print("\nAge Quartiles:", df['age_group'].value_counts().sort_index())
    print("\nRace:", df['race'].value_counts())
    print("\nLanguage:", df['language'].value_counts())
    print("\nEthnicity:", df['ethnicity'].value_counts())
    print("\nMarital Status:", df['maritalstatus'].value_counts())

## 4. Export Official Train, Validation, and Test Splits

In [ ]:
if not df.empty:
    train_df = df[df['use'] == 'training'].copy()
    val_df = df[df['use'] == 'validation'].copy()
    test_df = df[df['use'] == 'test'].copy()
    
    print(f"Train samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")
    
    train_out = os.path.join(DATA_DIR, "train.csv")
    val_out = os.path.join(DATA_DIR, "val.csv")
    test_out = os.path.join(DATA_DIR, "test.csv")
    
    train_df.to_csv(train_out, index=False)
    val_df.to_csv(val_out, index=False)
    test_df.to_csv(test_out, index=False)
    
    print(f"\nSaved: {train_out}")
    print(f"Saved: {val_out}")
    print(f"Saved: {test_out}")
    print("\nPreprocessing completed successfully!")